In [1]:
import torch

import triton
import triton.language as tl
from triton.runtime import driver

DEVICE = triton.runtime.driver.active.get_active_torch_device()

In [2]:
DEVICE

device(type='cuda', index=0)

In [3]:

def is_hip():
    return triton.runtime.driver.active.get_current_target().backend == "hip"



def is_cdna():
    return is_hip() and triton.runtime.driver.active.get_current_target().arch in ('gfx940', 'gfx941', 'gfx942',
                                                                                   'gfx90a', 'gfx908')
def naive_softmax(x):
    """Compute row-wise softmax of X using native pytorch

    We subtract the maximum element in order to avoid overflows. Softmax is invariant to
    this shift.
    """

    """
    なぜ私はnaive_softmaxの関数を作ろうとおもったのが。僕が今ときたいとおもっている課題は何か。
    まあ motivationはよくわからんけど、単純な加算をやりたくなったのはまちがいなさそうだ。
    """
    # xはたぶんtorchなんでしょう

    # read  MN elements ; write M  elements
    x_max = x.max(dim=1)[0]
    # read MN + M elements ; write MN elements
    # readとかwriteとかが何をやっているかはわからないが、標準化していることはわかる
    z = x - x_max[:, None]
    # read  MN elements ; write MN elements
    numerator = torch.exp(z)
    # read  MN elements ; write M  elements
    denominator = numerator.sum(dim=1)

    # read MN + M elements ; write MN elements
    ret = numerator / denominator[:, None]
    # in total: read 5MN + 2M elements ; wrote 3MN + 2M elements
    return ret

    

    

In [4]:
"""
torchは演算ごとに独立したkernelを起動して、

torchが自前のkernelをもっていて、DRAM に書き込んでいる=write M elements
DRAMは情報を保持している、SRAMは速いが情報を保持していない。
この例では、（reading 5⁢𝑀⁢𝑁 +2⁢𝑀 elements from DRAM and writing back 3⁢𝑀⁢𝑁 +2⁢𝑀 elements.）　分のioが発生している。
チップ上にのるような小さな計算であれば、最適化できそうだなって感じが今回のtutorial.
"""

'\ntorchは演算ごとに独立したkernelを起動して、\n\ntorchが自前のkernelをもっていて、DRAM に書き込んでいる=write M elements\nDRAMは情報を保持している、SRAMは速いが情報を保持していない。\nこの例では、（reading 5\u2062𝑀\u2062𝑁 +2\u2062𝑀 elements from DRAM and writing back 3\u2062𝑀\u2062𝑁 +2\u2062𝑀 elements.）\u3000分のioが発生している。\nチップ上にのるような小さな計算であれば、最適化できそうだなって感じが今回のtutorial.\n'

In [5]:
"""
予測をたてるべき

全体を分子として、8MN + 4M(要素) の読み書き(メモリトラフィック)が発生する
最初に読み込む1回と、書き込む1回は固定なので、2MNはfused版の総トラフィック

なのでこれを計算して、　(8MN + 4M)/2MN = 4 + 2/N (N が大きくなるほど、~4xに近づく、となる)
帯域律速なら、~4xに近づくことが期待できる
"""

'\n予測をたてるべき\n\n全体を分子として、8MN + 4M(要素) の読み書き(メモリトラフィック)が発生する\n最初に読み込む1回と、書き込む1回は固定なので、2MNはfused版の総トラフィック\n\nなのでこれを計算して、\u3000(8MN + 4M)/2MN = 4 + 2/N (N が大きくなるほど、~4xに近づく、となる)\n帯域律速なら、~4xに近づくことが期待できる\n'

In [9]:
@triton.jit
def softmax_kernel(output_ptr, input_ptr, input_row_stride, output_row_stride, n_rows, n_cols, BLOCK_SIZE: tl.constexpr,num_stages: tl.constexpr):
    # programはthread的なものって考えよう
    # なんで0??
    row_start = tl.program_id(0)
    row_step = tl.num_programs(0)
    tl.device_print("hello")
    for row_idx in tl.range(row_start, n_rows, row_step, num_stages=num_stages):
        # The stride represents how much we need to increase the pointer to advance 1 row
        row_start_ptr = input_ptr + row_idx * input_row_stride
        tl.device_print("row_idx", row_idx)
        





    

In [10]:
properties = driver.active.utils.get_device_properties(DEVICE.index)
SIZE_SMEM = properties["max_shared_mem"]


def softmax(x):
    n_rows, n_cols = x.shape
    print(n_rows, n_cols)
    # BLOCK_SIZEをもとめる.xの列数より大きい最小の二のべき乗を採用する
    BLOCK_SIZE = triton.next_power_of_2(n_cols)

    # なんでそれを二のべき乗に??
    # 二のべき乗である必要として、tritonが仕様として持っている制限がある
    # 全体のなんかの制限を求めるというか、BLOCK_SIZEだから処理の最小単位よね。
    # なら、最小のべき乗を最小するのはわかる。が、xの列数がなぜそこで出てくるんだろう。
    print(BLOCK_SIZE)

    # Another trick we can use is to ask the compiler to use more threads per row by
    # increasing the number of warps (`num_warps`) over which each row is distributed.
    # You will see in the next tutorial how to auto-tune this value in a more natural
    # way so you don't have to come up with manual heuristics yourself.
    # データによってここは変わる変数となる。今回は練習のために決め打ちなだけ
    num_warps = 8

    # Number of software pipelining stages.
    # 空き領域のチェックかな
    num_stages = 4 if SIZE_SMEM > 200000 else 2
    print(num_stages)

    # Allocate output
    y = torch.empty_like(x)

    kernel = softmax_kernel.warmup(  # pyright: ignore[reportFunctionMemberAccess]
        y,
        x,
        x.stride(0),
        y.stride(0),
        n_rows,
        n_cols,
        BLOCK_SIZE=BLOCK_SIZE,
        num_stages=num_stages,
        num_warps=num_warps,
        grid=(1,)
    )
        

    
    

In [11]:
torch.manual_seed(0)
x = torch.randn(1823, 781, device=DEVICE)
y_triton = softmax(x)
# y_torch = torch.softmax(x, axis=1)
# assert torch.allclose(y_triton, y_torch), (y_triton, y_torch)

1823 781
1024
2
